# 03 - Split Data, Encode Label, dan Standardisasi Nutrisi

**Tujuan:** Membagi data menjadi train/val/test dan mengubah label menjadi angka yang siap dipakai model.

| | |
|---|---|
| **Input** | `data/processed/manifest.csv` |
| **Output** | `data/processed/{train,val,test}.csv` dan `artifacts/labels.json` |
| **Prasyarat** | Notebook 02 |

> Setiap notebook berdiri sendiri: ia membaca hasil notebook sebelumnya dari Google Drive, jadi sesi Colab boleh diputus/diulang di antara notebook.

In [1]:
# --- Bootstrap (sama di setiap notebook) ---------------------------------------
# 1) Sambungkan Google Drive (tempat semua hasil disimpan permanen)
import os, sys, zipfile
from google.colab import drive
drive.mount('/content/drive')

# 2) Lokasi di Drive: folder ini sekaligus ROOT proyek (zip dan semua hasil ada di sini)
ROOT_ON_DRIVE = "/content/drive/MyDrive/models-train/mmfood-multitask"
ZIP_ON_DRIVE  = f"{ROOT_ON_DRIVE}/mmfood-multitask.zip"

#    Folder hasil = ROOT di Drive. Harus diset SEBELUM mengimpor config
os.environ["MMFOOD_PROJECT"] = ROOT_ON_DRIVE

# 3) Ekstrak proyek dari zip ke disk lokal Colab (hanya sekali per sesi)
REPO = "/content/mmfood-multitask"
if not os.path.exists(REPO):
    with zipfile.ZipFile(ZIP_ON_DRIVE) as z:
        z.extractall("/content")

# 4) Daftarkan folder src/ agar modul proyek bisa di-import
sys.path.insert(0, f"{REPO}/src")

# 5) Muat path folder (P) dan konfigurasi (CFG) dari src/config.py
from config import get_paths, load_cfg
P = get_paths()       # P.processed, P.checkpoints, P.hf_export, ... (folder dibuat otomatis)
CFG = load_cfg(P)     # nilai bawaan + perubahan dari notebook 00
print("Folder hasil:", P.project)

Mounted at /content/drive
Folder hasil: /content/drive/MyDrive/models-train/mmfood-multitask


## Langkah 3.1 - Split 80 / 10 / 10
Split **stratified berdasarkan `food_type`** agar kelas kecil muncul di ketiga bagian. Test set tidak disentuh sampai evaluasi akhir (notebook 05).

In [2]:
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from data_utils import NUTRITION_KEYS

# Muat daftar gambar yang berhasil diunduh
sub = pd.read_csv(f"{P.processed}/manifest.csv")

# 80% train, 20% sisa -> dibagi dua jadi val dan test
train_df, tmp = train_test_split(sub, test_size=0.2, stratify=sub.food_type, random_state=CFG["seed"])
val_df, test_df = train_test_split(tmp, test_size=0.5, stratify=tmp.food_type, random_state=CFG["seed"])
train_df, val_df, test_df = [d.reset_index(drop=True).copy() for d in (train_df, val_df, test_df)]
print("train/val/test:", len(train_df), len(val_df), len(test_df))

train/val/test: 28798 3600 3600


## Langkah 3.2 - Susun daftar kelas (label map)
Daftar kelas **hanya dibuat dari train** (val/test tidak boleh mempengaruhi). Metode masak yang jarang (`< min_per_cooking`) digabung ke `other`.

In [3]:
# Kelas food_type dan dish_name
food_labels = sorted(sub.food_type.unique().tolist())
dish_labels = sorted(sub.dish_label.dropna().unique().tolist())

# Metode masak: simpan yang cukup sering muncul di train, sisanya -> 'other'
cc = train_df.cooking_norm.value_counts()
cook_keep = cc[cc >= CFG["min_per_cooking"]].index.tolist()
for d in (train_df, val_df, test_df):
    d["cooking_final"] = d.cooking_norm.map(lambda c: c if c in cook_keep else "other")
cook_labels = sorted(train_df.cooking_final.unique().tolist())

print("food_type     :", food_labels)
print("cooking_method:", cook_labels)
print("dish_name     :", len(dish_labels), "kelas")

food_type     : ['Homemade food', 'Others', 'Packaged food', 'Raw vegetables and fruits', 'Restaurant food']
cooking_method: ['bake', 'boil', 'fry', 'grill', 'other', 'pan-fry', 'raw', 'simmer-stew', 'steam', 'stir-fry', 'unknown']
dish_name     : 50 kelas


## Langkah 3.3 - Ubah label teks menjadi angka
Baris yang `dish_name`-nya di luar Top-K diberi `-100`: nilai khusus yang **diabaikan** oleh loss klasifikasi dish (baris itu tetap dipakai untuk head lain).

In [4]:
# Peta teks -> indeks
f2i = {k: i for i, k in enumerate(food_labels)}
d2i = {k: i for i, k in enumerate(dish_labels)}
c2i = {k: i for i, k in enumerate(cook_labels)}

for d in (train_df, val_df, test_df):
    d["y_food"] = d.food_type.map(f2i)
    # dish di luar Top-K (NaN) -> -100 supaya diabaikan oleh CrossEntropyLoss
    d["y_dish"] = d.dish_label.map(lambda x: d2i.get(x, -100) if isinstance(x, str) else -100)
    d["y_cook"] = d.cooking_final.map(c2i)

print("Baris train dengan label dish:", int((train_df.y_dish != -100).sum()), "dari", len(train_df))

Baris train dengan label dish: 6464 dari 28798


## Langkah 3.4 - Standardisasi nutrisi
Nutrisi condong ke kanan, jadi dipakai `log1p` lalu dikurangi mean dan dibagi std. **Mean/std dihitung dari train saja** (mencegah kebocoran informasi dari val/test). Nilai asli dikembalikan saat inferensi dengan `expm1(z * std + mean)`.

In [5]:
# Statistik dari train (ruang log1p)
log_tr = np.log1p(train_df[NUTRITION_KEYS].values)
nut_mean, nut_std = log_tr.mean(0), log_tr.std(0) + 1e-6

# Terapkan ke ketiga split; hasilnya disimpan di kolom z_<nama>
for d in (train_df, val_df, test_df):
    z = (np.log1p(d[NUTRITION_KEYS].values) - nut_mean) / nut_std
    for i, k in enumerate(NUTRITION_KEYS):
        d[f"z_{k}"] = z[:, i]

print("mean (log1p):", nut_mean.round(3), "| std:", nut_std.round(3))

mean (log1p): [5.803 2.698 2.545 3.377] | std: [0.745 1.035 1.019 1.036]


## Langkah 3.5 - Simpan split dan label map

In [6]:
# Simpan tiga split untuk notebook 04-06
for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    d.to_csv(f"{P.processed}/{name}.csv", index=False)

# Simpan semua pemetaan label + statistik nutrisi (dipakai training, inferensi, dan model card)
LABELS = dict(food_type=food_labels, dish_name=dish_labels, cooking_method=cook_labels,
              nutrition_keys=NUTRITION_KEYS,
              nutrition_mean=nut_mean.tolist(), nutrition_std=nut_std.tolist())
json.dump(LABELS, open(f"{P.artifacts}/labels.json", "w"), indent=2)
print("Tersimpan di:", P.processed, "dan", f"{P.artifacts}/labels.json")

Tersimpan di: /content/drive/MyDrive/models-train/mmfood-multitask/data/processed dan /content/drive/MyDrive/models-train/mmfood-multitask/artifacts/labels.json


In [7]:
# Ringkasan distribusi food_type per split (periksa bahwa stratifikasi bekerja)
pd.DataFrame({"train": train_df.food_type.value_counts(),
              "val": val_df.food_type.value_counts(),
              "test": test_df.food_type.value_counts()})

,train,val,test
food_type,,,
Homemade food,13413,1676,1677
Restaurant food,10217,1277,1277
Raw vegetables and fruits,2696,337,337
Packaged food,2395,300,299
Others,77,10,10
